# Colab runner

Launcher only — no method code lives here. See `model.py`.

**Connect first:** `Select Kernel` → `Colab` → `New Colab Server` → pick **GPU**.
Then run these cells top to bottom.

Everything below executes on the Colab VM, not on your laptop.


## 1. Confirm we actually got a GPU

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

## 2. Mount Drive

(Or use the command palette: `Colab: Mount Google Drive to Server...`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Check the dataset paths resolve *before* training

Quotes matter — these folder names contain spaces.

In [ ]:
!ls "/content/drive/MyDrive/Research/Dataset New/final Real" | wc -l
!ls "/content/drive/MyDrive/Research/Dataset New/Fake" | wc -l

## 4. Clone the repo onto the VM

`/content` is wiped when the runtime dies, so this runs every fresh session.

In [ ]:
%cd /content
!git clone https://github.com/Dev-Joyson/CNN-based-GAN-face-detection.git 2>/dev/null || (cd CNN-based-GAN-face-detection && git pull)
%cd /content/CNN-based-GAN-face-detection
!pip install -q pyyaml

## 5. Sanity check: do the guard tests pass on this machine?

~3 seconds, no dataset needed.

In [ ]:
!pytest tests -q

## 6. Train

Epoch 1 is slow — it reads every image off Drive and writes the ~10 GB cache to
local disk. Later epochs read the cache and are much faster. Don't kill it.

Outputs go to Drive (`out_dir` in the config), so they survive a disconnect.

In [ ]:
!python train.py --config configs/test13_face.yaml

## 7. Background-only control run

Reuses the cache only if `limit_per_class` matches; test14 uses 20k so it builds its own.

In [ ]:
!python train.py --config configs/test14_background.yaml

## 8. Plot the curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv("/content/drive/MyDrive/Research/experiments/test13_face/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h[["accuracy", "val_accuracy"]].plot(ax=ax[0], title="accuracy")
h[["auc", "val_auc"]].plot(ax=ax[1], title="AUC")
plt.tight_layout(); plt.show()